%md
# 🚀 Instrucciones de uso del notebook

## Requisitos previos

Antes de ejecutar el notebook, asegúrate de contar con un archivo que contenga las cédulas a consultar. El programa admite archivos de texto (`.txt`) con una cédula por línea o archivos CSV (`.csv`) 

**Ejemplo de archivo de texto (.txt):**

```text
1234567890
9876543210
1122334455
```

**Ejemplo de archivo CSV (.csv):**

```csv
1234567890
9876543210
1122334455
```

El archivo de entrada debe ubicarse en la ruta configurada dentro del notebook:

```text
/Volumes/datacedulas/bronze/zona_aterrizaje/
```

---

## Ejecución del notebook

1. Verifica que el archivo de entrada exista en la ruta configurada.
2. Ejecuta las celdas del notebook en el orden establecido.
3. El programa leerá automáticamente las cédulas del archivo.
4. Posteriormente realizará una consulta individual a la API de la Registraduría para cada documento.
5. Los resultados serán almacenados automáticamente en un archivo CSV.

---

## Reanudación del proceso

Si la ejecución se interrumpe por alcanzar el tiempo máximo permitido o por cualquier otra causa, solo debes volver a ejecutar la celda de ejecución. El programa continuará automáticamente desde la última cédula procesada gracias al archivo de progreso.

---

## Archivo de salida

Los resultados se almacenan en:

```text
/Volumes/datacedulas/bronze/zona_aterrizaje/salidascedula.csv
```

Cada registro contiene la siguiente información:

- Cédula consultada.
- Resultado obtenido de la API.
- Fecha y hora de la consulta.

**Ejemplo:**

```csv
cedula,resultado,fecha_consulta
1234567890,VIGENTE,2026-06-25 14:32:10
9876543210,CANCELADA POR MUERTE,2026-06-25 14:32:15
1122334455,ERROR: HTTP 500,2026-06-25 14:32:20
```

---

## Consideraciones importantes

- El programa espera **4 segundos** entre consultas para evitar saturar la API.
- Cada solicitud tiene un tiempo máximo de espera de **10 segundos**.
- El progreso se guarda después de procesar cada cédula.
- Si ocurre un error durante una consulta, este queda registrado en el archivo de resultados y el procesamiento continúa con la siguiente cédula.
- Cuando todas las consultas finalizan correctamente, el archivo temporal de progreso se elimina automáticamente.



# Bloque 1. Preparación del entorno y definición de funciones

En este bloque se prepara el entorno de trabajo en **Databricks** instalando la librería `requests`, necesaria para realizar consultas a la API de la Registraduría. Posteriormente, se importan las librerías utilizadas durante el desarrollo, como las encargadas del manejo de archivos, fechas y tiempos de espera.

También se definen los parámetros generales del programa, incluyendo las rutas de los archivos de entrada y salida, la dirección de la API, el tiempo máximo de ejecución y el intervalo entre consultas.

Finalmente, se crean las funciones que permiten leer las cédulas desde un archivo de texto o CSV, administrar el progreso de la ejecución, almacenar los resultados y ejecutar la lógica principal encargada de consultar el estado de cada documento y registrar la respuesta obtenida.

In [0]:
%pip install requests --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Bloque 2. Configuración de rutas y parámetros de ejecución

En este bloque se actualizan las rutas de trabajo para utilizar los archivos almacenados dentro de los **Volumes de Databricks**. Aquí se define el archivo que contiene las cédulas a consultar y el archivo CSV donde se almacenarán los resultados del proceso.

Además, se muestran en pantalla los parámetros principales de la ejecución, como la ubicación de los archivos, la dirección de la API, el tiempo máximo permitido para la ejecución y el tiempo de espera entre cada consulta. Esto permite verificar que toda la configuración sea correcta antes de iniciar el procesamiento.

In [0]:
import requests
import time
import csv
import os
from datetime import datetime, timedelta

# ========== CONFIGURACIÓN ==========
ARCHIVO_ENTRADA = "/Workspace/Users/andres.suarez0319@unaula.edu.co/lista_medioscomunicacion/documentos de identidad.txt"  # 👈 Archivo con las cédulas
ARCHIVO_SALIDA = "/Workspace/Users/andres.suarez0319@unaula.edu.co/resultados_cedulas.csv"  # Archivo donde se guardan los resultados
PROGRESS_FILE = "/Workspace/Users/andres.suarez0319@unaula.edu.co/progreso_cedulas.txt"
API_URL = "https://defunciones.registraduria.gov.co:8443/VigenciaCedula/consulta"
TIMEOUT_MINUTOS = 29
ESPERA_ENTRE_CONSULTAS = 4  # segundos

# Función para leer cédulas del archivo
def leer_cedulas():
    """
    Lee las cédulas del archivo de entrada.
    Soporta archivos .txt (una cédula por línea) y .csv (columna 'cedula').
    Retorna una lista de cédulas.
    """
    cedulas = []
    
    if not os.path.exists(ARCHIVO_ENTRADA):
        print(f"❌ Error: No se encontró el archivo {ARCHIVO_ENTRADA}")
        return cedulas
    
    try:
        # Determinar el formato del archivo por su extensión
        extension = ARCHIVO_ENTRADA.split('.')[-1].lower()
        
        if extension == 'csv':
            # Leer archivo CSV
            with open(ARCHIVO_ENTRADA, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    cedula = str(row.get('cedula', '')).strip()
                    if cedula and cedula.lower() not in ['cedula', 'nuip', 'documento']:
                        cedulas.append(cedula)
        else:
            # Leer archivo de texto (una cédula por línea)
            with open(ARCHIVO_ENTRADA, 'r', encoding='utf-8') as f:
                for linea in f:
                    linea = linea.strip()
                    
                    # Ignorar líneas vacías y comentarios
                    if not linea or linea.startswith('#'):
                        continue
                    
                    # Si la línea tiene formato "número: cédula", extraer solo la cédula
                    if ':' in linea:
                        partes = linea.split(':', 1)
                        if len(partes) == 2:
                            cedula = partes[1].strip()
                            if cedula:  # Solo agregar si hay cédula después del :
                                cedulas.append(cedula)
                    else:
                        # Formato simple: una cédula por línea
                        cedula = linea
                        if cedula:
                            cedulas.append(cedula)
        
        print(f"✅ Se leyeron {len(cedulas)} cédulas del archivo")
        return cedulas
        
    except Exception as e:
        print(f"❌ Error al leer el archivo: {e}")
        return cedulas

# Función para obtener el último índice procesado
def obtener_ultimo_indice():
    """
    Lee el último índice procesado desde el archivo de progreso.
    Si no existe, retorna 0 (primera cédula).
    """
    try:
        with open(PROGRESS_FILE, 'r') as f:
            indice = int(f.read().strip())
            print(f"📍 Reanudando desde cédula #{indice + 1}")
            return indice
    except FileNotFoundError:
        print("📍 Iniciando desde el principio")
        return 0
    except Exception as e:
        print(f"⚠️ Error al leer progreso: {e}. Iniciando desde el principio.")
        return 0

# Función para guardar el progreso
def guardar_progreso(indice):
    """
    Guarda el índice actual en el archivo de progreso.
    """
    try:
        with open(PROGRESS_FILE, 'w') as f:
            f.write(str(indice))
    except Exception as e:
        print(f"⚠️ Error al guardar progreso: {e}")

# Función para inicializar el archivo de salida
def inicializar_archivo_salida():
    """
    Crea el archivo CSV de salida con los encabezados si no existe.
    """
    if not os.path.exists(ARCHIVO_SALIDA):
        try:
            with open(ARCHIVO_SALIDA, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['cedula', 'resultado', 'fecha_consulta'])
            print(f"✅ Archivo de salida creado: {ARCHIVO_SALIDA}")
        except Exception as e:
            print(f"⚠️ Error al crear archivo de salida: {e}")

# Función para guardar resultado
def guardar_resultado(cedula, resultado):
    """
    Guarda el resultado de una consulta en el archivo CSV de salida.
    """
    try:
        fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(ARCHIVO_SALIDA, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([cedula, resultado, fecha_consulta])
    except Exception as e:
        print(f"⚠️ Error al guardar resultado: {e}")

print("✅ Configuración cargada correctamente")

✅ Configuración cargada correctamente


# Bloque 3. Ejecución del proceso de consulta

En este bloque se invoca la función principal `consultar_estado_cedulas()`, encargada de ejecutar todo el proceso de forma automática. La función lee las cédulas almacenadas en el archivo de entrada, realiza una consulta individual a la API de la Registraduría para cada documento y obtiene su estado de vigencia.

Durante la ejecución, el programa registra el progreso, controla posibles errores de conexión o respuesta y guarda inmediatamente cada resultado en un archivo CSV, permitiendo continuar el proceso desde el último punto procesado en caso de interrupción.


In [0]:
def consultar_estado_cedulas():
    """
    Función principal que:
    1. Lee las cédulas del archivo de entrada
    2. Consulta el estado de cada cédula en la API de la Registraduría
    3. Guarda los resultados en un archivo CSV
    4. Guarda el progreso después de cada consulta
    5. Se detiene después de 29 minutos
    """
    
    # Leer cédulas del archivo
    cedulas = leer_cedulas()
    
    if not cedulas:
        print("❌ No se encontraron cédulas para procesar")
        print(f"   Asegúrate de que existe el archivo: {ARCHIVO_ENTRADA}")
        return
    
    print(f"📋 Total de cédulas a procesar: {len(cedulas)}")
    
    # Inicializar archivo de salida
    inicializar_archivo_salida()
    
    # Obtener el último índice procesado
    ultimo_indice = obtener_ultimo_indice()
    
    # Hora de inicio para controlar el timeout
    inicio = datetime.now()
    limite_tiempo = timedelta(minutes=TIMEOUT_MINUTOS)
    
    cedulas_procesadas = 0
    cedulas_exitosas = 0
    cedulas_error = 0
    
    print(f"\n🚀 Iniciando procesamiento...\n")
    
    # Iterar sobre las cédulas desde el último índice procesado
    for i in range(ultimo_indice, len(cedulas)):
        # Verificar timeout
        tiempo_transcurrido = datetime.now() - inicio
        if tiempo_transcurrido >= limite_tiempo:
            print(f"\n⏸️ Tiempo límite de {TIMEOUT_MINUTOS} minutos alcanzado.")
            print(f"📊 Estadísticas: {cedulas_procesadas} procesadas | {cedulas_exitosas} exitosas | {cedulas_error} con error")
            print(f"\n📄 Resultados guardados en: {ARCHIVO_SALIDA}")
            return
        
        cedula = cedulas[i]
        
        print(f"🔍 Cédula {i + 1}/{len(cedulas)}: {cedula}")
        
        # Preparar la solicitud a la API
        payload = {"nuip": cedula}
        headers = {"Content-Type": "application/json"}
        
        try:
            # Hacer la consulta a la API
            response = requests.post(
                API_URL, 
                json=payload, 
                headers=headers,
                timeout=10  # Timeout de 10 segundos para la solicitud
            )
            
            # Verificar si la respuesta fue exitosa
            if response.status_code == 200:
                resultado = response.json()
                vigencia = resultado.get('vigencia', 'SIN RESPUESTA')
                
                # Guardar el resultado en el archivo CSV
                guardar_resultado(cedula, vigencia)
                
                print(f"   ✅ Resultado: {vigencia}")
                cedulas_exitosas += 1
            else:
                error_msg = f"HTTP {response.status_code}"
                guardar_resultado(cedula, f"ERROR: {error_msg}")
                print(f"   ❌ Error: {error_msg}")
                cedulas_error += 1
                
        except requests.exceptions.Timeout:
            error_msg = "TIMEOUT"
            guardar_resultado(cedula, f"ERROR: {error_msg}")
            print(f"   ❌ Error: Timeout en la solicitud")
            cedulas_error += 1
            
        except requests.exceptions.RequestException as e:
            error_msg = str(e)[:50]  # Limitar longitud del mensaje
            guardar_resultado(cedula, f"ERROR: {error_msg}")
            print(f"   ❌ Error en la solicitud: {e}")
            cedulas_error += 1
            
        except Exception as e:
            error_msg = str(e)[:50]
            guardar_resultado(cedula, f"ERROR: {error_msg}")
            print(f"   ❌ Error inesperado: {e}")
            cedulas_error += 1
        
        # Guardar el progreso
        guardar_progreso(i + 1)
        cedulas_procesadas += 1
        
        # Esperar entre consultas para no saturar la API
        time.sleep(ESPERA_ENTRE_CONSULTAS)
    
    # Si llegó aquí, terminó de procesar todas las cédulas
    print(f"\n✅ Procesamiento completado!")
    print(f"📊 Estadísticas finales: {cedulas_procesadas} procesadas | {cedulas_exitosas} exitosas | {cedulas_error} con error")
    print(f"📄 Resultados guardados en: {ARCHIVO_SALIDA}")
    
    # Limpiar el archivo de progreso
    try:
        if os.path.exists(PROGRESS_FILE):
            os.remove(PROGRESS_FILE)
            print("🗑️ Archivo de progreso eliminado")
    except Exception as e:
        print(f"⚠️ No se pudo eliminar el archivo de progreso: {e}")

print("✅ Función consultar_estado_cedulas() definida")

✅ Función consultar_estado_cedulas() definida


# Bloque 4. Resultados y finalización del proceso

Una vez finalizadas todas las consultas, el programa presenta un resumen con la cantidad de cédulas procesadas, las consultas exitosas y los posibles errores encontrados durante la ejecución.

Finalmente, si todas las cédulas fueron consultadas correctamente, elimina el archivo temporal de progreso y deja disponible el archivo CSV con los resultados finales para su posterior análisis o utilización.


In [0]:
# ========================================
# EJECUTAR CONSULTA DE ESTADO DE CÉDULAS
# ========================================

# Sobrescribir las rutas de los archivos con las de Volumes
ARCHIVO_ENTRADA = "/Volumes/datacedulas/bronze/zona_aterrizaje/documentos de identidad.txt"
ARCHIVO_SALIDA = "/Volumes/datacedulas/bronze/zona_aterrizaje/salidascedula.csv"
PROGRESS_FILE = "/Workspace/Users/andres.suarez0319@unaula.edu.co/progreso_cedulas.txt"

print("🚀 Iniciando consulta de estado de cédulas...\n")
print("="*60)
print(f"Archivo de entrada: {ARCHIVO_ENTRADA}")
print(f"Archivo de salida: {ARCHIVO_SALIDA}")
print(f"API: {API_URL}")
print(f"Timeout: {TIMEOUT_MINUTOS} minutos")
print(f"Espera entre consultas: {ESPERA_ENTRE_CONSULTAS} segundos")
print("="*60)
print()

# Ejecutar la función principal
consultar_estado_cedulas()

🚀 Iniciando consulta de estado de cédulas...

Archivo de entrada: /Volumes/datacedulas/bronze/zona_aterrizaje/documentos de identidad.txt
Archivo de salida: /Volumes/datacedulas/bronze/zona_aterrizaje/salidascedula.csv
API: https://defunciones.registraduria.gov.co:8443/VigenciaCedula/consulta
Timeout: 29 minutos
Espera entre consultas: 4 segundos

✅ Se leyeron 22 cédulas del archivo
📋 Total de cédulas a procesar: 22
📍 Iniciando desde el principio

🚀 Iniciando procesamiento...

🔍 Cédula 1/22: 1152440319
⚠️ Error al guardar resultado: [Errno 29] Illegal seek
   ✅ Resultado: Vigente (Vivo)
🔍 Cédula 2/22: 6789531
⚠️ Error al guardar resultado: [Errno 29] Illegal seek
   ✅ Resultado: Vigente (Vivo)
🔍 Cédula 3/22: 98592244
⚠️ Error al guardar resultado: [Errno 29] Illegal seek
   ✅ Resultado: Vigente (Vivo)
🔍 Cédula 4/22: 43268194
⚠️ Error al guardar resultado: [Errno 29] Illegal seek
   ✅ Resultado: Vigente (Vivo)
🔍 Cédula 5/22: 44006980
⚠️ Error al guardar resultado: [Errno 29] Illegal see

In [0]:
# Leer todos los archivos de la zona de aterrizaje como DataFrame
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/datacedulas/bronze/zona_aterrizaje/*")

# Escribir el DataFrame en formato Delta en el esquema bronze
df.write.format("delta").mode("overwrite").saveAsTable("bronze.zona_aterrizaje")